# Day 3 — ML Modeling, Evaluation & SHAP

This notebook picks up from the Day 2 Delta table (`capstone.bronze.claims_engineered`).

**What Day 2 already completed:**
- Raw CSV ingestion, recoding (PotentialFraud, ChronicCond_*, RenalDiseaseIndicator)
- Vertical UNION of Inpatient/Outpatient with `claim_type` flag
- Beneficiary + Train label joins
- Imputation (DeductibleAmtPaid median, ClmDiagnosisCode_1 UNKNOWN fill)
- Claim-level feature engineering (ClaimDuration, LengthOfStay, Age, IsDeceased, DiagnosisCodeCount, ProcedureCodeCount, PhysicianCount, HasOperatingPhysician, HasOtherPhysician, ChronicConditionCount, HasPrimaryDiagnosis, ICD_Chapter, MedicareCoverageMonths, IPtoOPReimbursementRatio)
- Class weight computation
- Column drops + Delta save

**What this notebook does:**
1. Provider-level train/test split (leakage-safe)
2. Provider aggregate features computed on training set only
3. ClaimAmtZScore with proper zero-handling
4. Targeted null handling (LengthOfStay sentinel, not blanket fillna)
5. One-hot encoding (fitted on train, applied to test)
6. Logistic Regression + Decision Tree training with class weights
7. Evaluation (ROC-AUC, PR-AUC, F1, threshold tuning)
8. SHAP value computation
9. MLflow logging, artifact export, model registration

In [ ]:
import mlflow
import mlflow.sklearn
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, auc,
    f1_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import OneHotEncoder
import shap
import json
import warnings
warnings.filterwarnings("ignore")

spark = SparkSession.builder.getOrCreate()

# ── Configure MLflow experiment ──
EXPERIMENT_NAME = "/Users/<your-user>/predictive-denial-prevention"
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment: {EXPERIMENT_NAME}")

## 1. Load Day 2 Delta Table

In [ ]:
claims_df = spark.read.table("capstone.bronze.claims_engineered")

print(f"Loaded: {claims_df.count():,} rows, {len(claims_df.columns)} columns")
claims_df.printSchema()

## 2. Data Validation

In [ ]:
# Verify critical columns and recodes
assert claims_df.filter(F.col("PotentialFraud").isNull()).count() == 0, \
    "Null PotentialFraud labels found"
assert set(
    claims_df.select("PotentialFraud").distinct()
    .toPandas()["PotentialFraud"].tolist()
) <= {0, 1}, "PotentialFraud not binary"

# Class distribution
print("Class distribution:")
claims_df.groupBy("PotentialFraud").count().show()

# Verify LengthOfStay is null (not 0) for outpatient rows.
# Day 2 computes LengthOfStay = datediff(DischargeDt, AdmissionDt),
# which correctly yields null when both dates are null (outpatient).
# If someone ran a blanket fillna(0) before the Delta save, this check catches it.
op_los_nulls = claims_df.filter(
    (F.col("claim_type") == 0) & F.col("LengthOfStay").isNull()
).count()
op_total = claims_df.filter(F.col("claim_type") == 0).count()
print(f"Outpatient LengthOfStay nulls: {op_los_nulls:,} / {op_total:,} "
      f"({'PASS' if op_los_nulls == op_total else 'FAIL — check Day 2'})")

# Verify Provider column exists (needed for the split below)
assert "Provider" in claims_df.columns, "Provider column missing — cannot do leakage-safe split"
print("\nAll validations passed.")

## 3. Provider-Level Train / Test Split (Leakage-Safe)

Provider aggregate features (ProviderAvgClaimAmt, ClaimAmtZScore, etc.) are derived from claims.
A naive random row-level split would leak provider billing history across train/test, inflating metrics.

**Strategy:** Split *providers* 80/20 (stratified on PotentialFraud), then assign ALL of a
provider's claims exclusively to one set. Provider aggregates are computed on training claims only.

In [ ]:
# Get one row per provider with its fraud label
providers = claims_df.select("Provider", "PotentialFraud").distinct()

# Stratified split: split fraud and non-fraud providers separately
fraud_providers = providers.filter(F.col("PotentialFraud") == 1)
clean_providers = providers.filter(F.col("PotentialFraud") == 0)

train_fraud, test_fraud = fraud_providers.randomSplit([0.8, 0.2], seed=42)
train_clean, test_clean = clean_providers.randomSplit([0.8, 0.2], seed=42)

train_provider_ids = train_fraud.select("Provider").union(
    train_clean.select("Provider"))
test_provider_ids = test_fraud.select("Provider").union(
    test_clean.select("Provider"))

# Assign every claim to train or test based on its provider
train_df = claims_df.join(
    F.broadcast(train_provider_ids), on="Provider", how="inner")
test_df = claims_df.join(
    F.broadcast(test_provider_ids), on="Provider", how="inner")

train_count = train_df.count()
test_count = test_df.count()
train_fraud_rate = train_df.filter(
    F.col("PotentialFraud") == 1).count() / train_count
test_fraud_rate = test_df.filter(
    F.col("PotentialFraud") == 1).count() / test_count

print(f"Train providers: {train_provider_ids.count():,}  |  "
      f"Test providers: {test_provider_ids.count():,}")
print(f"Train claims:    {train_count:,}  |  "
      f"Test claims:    {test_count:,}")
print(f"Train fraud %:   {train_fraud_rate:.3f}  |  "
      f"Test fraud %:   {test_fraud_rate:.3f}")

## 4. Provider Aggregate Features (Training Set Only)

Computed exclusively from training claims. The resulting lookup table is joined onto
**both** train and test, so test claims never contribute to their own provider statistics.
This same lookup table is the source at inference time (Day 4, Test Case 4).

In [ ]:
provider_stats = train_df.groupBy("Provider").agg(
    F.count("*").alias("ProviderClaimCount"),
    F.avg("InscClaimAmtReimbursed").alias("ProviderAvgClaimAmt"),
    F.stddev("InscClaimAmtReimbursed").alias("ProviderClaimAmtStdDev"),
    F.avg("DiagnosisCodeCount").alias("ProviderAvgDiagnosisCount"),
    F.avg("LengthOfStay").alias("ProviderAvgLOS")
)

# Join onto both sets
train_df = train_df.join(provider_stats, on="Provider", how="left")
test_df = test_df.join(provider_stats, on="Provider", how="left")

print(f"Provider lookup table: {provider_stats.count():,} providers")
provider_stats.show(5)

## 5. ClaimAmtZScore + Null Handling

Three fixes from the original notebook:

1. **ZScore:** Proper zero-handling (`when stddev > 0 ... otherwise 0.0`), not the `stddev + 1` hack
   which compresses every z-score and breaks the "3+ std devs = anomaly" interpretation.
2. **LengthOfStay:** Outpatient nulls filled with `-1` (sentinel), not `0`.
   `0` means "same-day discharge" (a real inpatient event); `-1` means "not applicable."
3. **Unseen test providers:** Stats filled with global training averages, not `0`.

In [ ]:
# Global training averages for fallback on unseen test providers
_global = train_df.select(
    F.avg("InscClaimAmtReimbursed").alias("g_avg_claim"),
    F.count("*").alias("g_count")
).collect()[0]

GLOBAL_AVG_CLAIM = float(_global["g_avg_claim"])
GLOBAL_COUNT = int(_global["g_count"])


def post_process(df):
    """Apply provider-stat fallbacks, z-score, and targeted null fills."""

    # ── Unseen-provider fallback ──
    df = (df
        .withColumn("ProviderClaimCount",
            F.coalesce(F.col("ProviderClaimCount"), F.lit(GLOBAL_COUNT)))
        .withColumn("ProviderAvgClaimAmt",
            F.coalesce(F.col("ProviderAvgClaimAmt"), F.lit(GLOBAL_AVG_CLAIM)))
        .withColumn("ProviderClaimAmtStdDev",
            F.coalesce(F.col("ProviderClaimAmtStdDev"), F.lit(0.0)))
    )

    # ── ClaimAmtZScore: divide by stddev, return 0 when stddev is 0 ──
    df = df.withColumn("ClaimAmtZScore",
        F.when(F.col("ProviderClaimAmtStdDev") > 0,
            (F.col("InscClaimAmtReimbursed") - F.col("ProviderAvgClaimAmt"))
            / F.col("ProviderClaimAmtStdDev")
        ).otherwise(0.0)
    )

    # ── LengthOfStay: -1 sentinel for outpatient, not 0 ──
    df = df.withColumn("LengthOfStay",
        F.when(F.col("LengthOfStay").isNull(), F.lit(-1))
         .otherwise(F.col("LengthOfStay")))

    # ── Targeted numeric null fills (only columns that can legitimately be 0) ──
    for c in ["DeductibleAmtPaid", "ClaimDuration", "Age",
              "DiagnosisCodeCount", "ProcedureCodeCount",
              "PhysicianCount", "ChronicConditionCount",
              "MedicareCoverageMonths", "IPtoOPReimbursementRatio"]:
        df = df.withColumn(c, F.coalesce(F.col(c), F.lit(0)))

    # ── Categorical null fills ──
    df = df.fillna({
        "Gender": "Unknown", "Race": "Unknown",
        "State": "Unknown",  "ICD_Chapter": "UNK"
    })

    return df


train_df = post_process(train_df)
test_df = post_process(test_df)

print("Post-processing complete.")
print(f"Train ClaimAmtZScore stats:")
train_df.select(
    F.min("ClaimAmtZScore"), F.avg("ClaimAmtZScore"),
    F.max("ClaimAmtZScore")
).show()

## 6. Feature Definitions

Matches the plan spec exactly. Provider intermediate columns (`ProviderClaimAmtStdDev`,
`ProviderAvgDiagnosisCount`, `ProviderAvgLOS`) are used for the lookup table and z-score
computation but are **not** direct model inputs.

In [ ]:
# ── Features that go into the model (plan spec) ──
NUMERIC_FEATURES = [
    "InscClaimAmtReimbursed", "DeductibleAmtPaid",
    "ClaimDuration", "LengthOfStay", "Age",
    "DiagnosisCodeCount", "ProcedureCodeCount",
    "PhysicianCount", "ChronicConditionCount",
    "MedicareCoverageMonths", "IPtoOPReimbursementRatio",
    "ProviderClaimCount", "ProviderAvgClaimAmt", "ClaimAmtZScore"
]

BINARY_FEATURES = [
    "claim_type", "IsDeceased", "HasRenalDisease",
    "HasOperatingPhysician", "HasOtherPhysician", "HasPrimaryDiagnosis"
]

CATEGORICAL_FEATURES = ["Gender", "Race", "State", "ICD_Chapter"]

LABEL = "PotentialFraud"
WEIGHT = "class_weight"

# ── Columns kept for tracking / joins but NOT fed into the model ──
ID_COLS = ["Provider", "ClaimID", "BeneID"]

print(f"Numeric:     {len(NUMERIC_FEATURES)}")
print(f"Binary:      {len(BINARY_FEATURES)}")
print(f"Categorical: {len(CATEGORICAL_FEATURES)} (will be one-hot encoded)")

## 7. Convert to Pandas & One-Hot Encode

Spark handles the distributed data processing (joins, aggregates, split).
For modeling + SHAP, we convert to pandas and use scikit-learn — the plan allows this
(`class_weight='balanced' if using Scikit-Learn`) and it gives clean SHAP integration
without needing Spark ML vector wrappers.

In [ ]:
# Select only needed columns, convert to pandas
select_cols = (
    ID_COLS + NUMERIC_FEATURES + BINARY_FEATURES
    + CATEGORICAL_FEATURES + [LABEL, WEIGHT]
)
select_cols = [c for c in select_cols if c in train_df.columns]

train_pd = train_df.select(select_cols).toPandas()
test_pd = test_df.select(select_cols).toPandas()

print(f"Train pandas: {train_pd.shape}  |  Test pandas: {test_pd.shape}")

# ── One-hot encoding: fit on train, transform both ──
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
ohe.fit(train_pd[CATEGORICAL_FEATURES])

train_cat = pd.DataFrame(
    ohe.transform(train_pd[CATEGORICAL_FEATURES]),
    columns=ohe.get_feature_names_out(CATEGORICAL_FEATURES),
    index=train_pd.index
)
test_cat = pd.DataFrame(
    ohe.transform(test_pd[CATEGORICAL_FEATURES]),
    columns=ohe.get_feature_names_out(CATEGORICAL_FEATURES),
    index=test_pd.index
)

# ── Assemble final feature matrices ──
X_train = pd.concat(
    [train_pd[NUMERIC_FEATURES + BINARY_FEATURES], train_cat], axis=1)
X_test = pd.concat(
    [test_pd[NUMERIC_FEATURES + BINARY_FEATURES], test_cat], axis=1)

y_train = train_pd[LABEL].astype(int)
y_test = test_pd[LABEL].astype(int)
w_train = train_pd[WEIGHT]

# Ensure no NaNs slipped through
assert X_train.isna().sum().sum() == 0, \
    f"NaNs in X_train: {X_train.columns[X_train.isna().any()].tolist()}"
assert X_test.isna().sum().sum() == 0, \
    f"NaNs in X_test: {X_test.columns[X_test.isna().any()].tolist()}"

print(f"\nFinal feature count: {X_train.shape[1]}")
print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"Positive class rate — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")

## 8. Train Models

In [ ]:
# ── Logistic Regression ──
with mlflow.start_run(run_name="logistic_regression") as lr_run:
    lr = LogisticRegression(max_iter=1000, random_state=42, solver="lbfgs")
    lr.fit(X_train, y_train, sample_weight=w_train)

    lr_proba = lr.predict_proba(X_test)[:, 1]
    lr_pred = lr.predict(X_test)

    lr_roc_auc = roc_auc_score(y_test, lr_proba)
    lr_prec, lr_rec, _ = precision_recall_curve(y_test, lr_proba)
    lr_pr_auc = auc(lr_rec, lr_prec)
    lr_f1 = f1_score(y_test, lr_pred)
    lr_cm = confusion_matrix(y_test, lr_pred)

    mlflow.log_params({
        "model_type": "LogisticRegression",
        "max_iter": 1000,
        "solver": "lbfgs",
        "n_features": X_train.shape[1],
        "split_strategy": "provider_level_stratified",
        "split_seed": 42,
    })
    mlflow.log_metrics({
        "roc_auc": lr_roc_auc,
        "pr_auc": lr_pr_auc,
        "f1_default_0.5": lr_f1,
    })
    mlflow.sklearn.log_model(lr, "model")
    mlflow.log_dict({"confusion_matrix": lr_cm.tolist()}, "confusion_matrix.json")

    lr_run_id = lr_run.info.run_id

print(f"Logistic Regression")
print(f"  ROC-AUC: {lr_roc_auc:.4f}  |  PR-AUC: {lr_pr_auc:.4f}  |  F1: {lr_f1:.4f}")
print(f"  Confusion Matrix:\n{lr_cm}")

In [ ]:
# ── Decision Tree ──
with mlflow.start_run(run_name="decision_tree") as dt_run:
    dt = DecisionTreeClassifier(max_depth=10, random_state=42)
    dt.fit(X_train, y_train, sample_weight=w_train)

    dt_proba = dt.predict_proba(X_test)[:, 1]
    dt_pred = dt.predict(X_test)

    dt_roc_auc = roc_auc_score(y_test, dt_proba)
    dt_prec, dt_rec, _ = precision_recall_curve(y_test, dt_proba)
    dt_pr_auc = auc(dt_rec, dt_prec)
    dt_f1 = f1_score(y_test, dt_pred)
    dt_cm = confusion_matrix(y_test, dt_pred)

    mlflow.log_params({
        "model_type": "DecisionTree",
        "max_depth": 10,
        "n_features": X_train.shape[1],
        "split_strategy": "provider_level_stratified",
        "split_seed": 42,
    })
    mlflow.log_metrics({
        "roc_auc": dt_roc_auc,
        "pr_auc": dt_pr_auc,
        "f1_default_0.5": dt_f1,
    })
    mlflow.sklearn.log_model(dt, "model")
    mlflow.log_dict({"confusion_matrix": dt_cm.tolist()}, "confusion_matrix.json")

    dt_run_id = dt_run.info.run_id

print(f"Decision Tree")
print(f"  ROC-AUC: {dt_roc_auc:.4f}  |  PR-AUC: {dt_pr_auc:.4f}  |  F1: {dt_f1:.4f}")
print(f"  Confusion Matrix:\n{dt_cm}")

## 9. Model Comparison & Threshold Tuning

PR-AUC is the primary selection metric (more informative than ROC-AUC under 10% class imbalance).
The default 0.5 cutoff is arbitrary — we sweep thresholds to find the F1-maximizing boundary,
which can then be adjusted by the business based on False Positive vs False Negative cost.

In [ ]:
import matplotlib.pyplot as plt

# ── Side-by-side comparison ──
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree"],
    "ROC-AUC": [lr_roc_auc, dt_roc_auc],
    "PR-AUC": [lr_pr_auc, dt_pr_auc],
    "F1 (0.5 cutoff)": [lr_f1, dt_f1],
})
print(results.to_string(index=False))

# Select best by PR-AUC
if lr_pr_auc >= dt_pr_auc:
    best_model = lr
    best_proba = lr_proba
    best_name = "LogisticRegression"
    best_run_id = lr_run_id
else:
    best_model = dt
    best_proba = dt_proba
    best_name = "DecisionTree"
    best_run_id = dt_run_id

print(f"\nBest model: {best_name} (PR-AUC = {max(lr_pr_auc, dt_pr_auc):.4f})")

# ── Threshold tuning ──
thresholds = np.arange(0.05, 0.95, 0.01)
f1_scores = []
for t in thresholds:
    preds_t = (best_proba >= t).astype(int)
    f1_scores.append(f1_score(y_test, preds_t, zero_division=0))

optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
optimal_f1 = f1_scores[optimal_idx]

print(f"Optimal threshold: {optimal_threshold:.2f}  →  F1: {optimal_f1:.4f}")

# ── PR curve plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(lr_rec, lr_prec, label=f"LogReg (PR-AUC={lr_pr_auc:.3f})")
axes[0].plot(dt_rec, dt_prec, label=f"DecTree (PR-AUC={dt_pr_auc:.3f})")
axes[0].axhline(y=y_test.mean(), color="gray", linestyle="--", label="Baseline (pos rate)")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curves")
axes[0].legend()

axes[1].plot(thresholds, f1_scores)
axes[1].axvline(x=optimal_threshold, color="red", linestyle="--",
                label=f"Optimal = {optimal_threshold:.2f}")
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("F1 Score")
axes[1].set_title(f"F1 vs Threshold ({best_name})")
axes[1].legend()

plt.tight_layout()
plt.show()

# Log tuned metrics to the best run
with mlflow.start_run(run_id=best_run_id):
    mlflow.log_param("optimal_threshold", float(optimal_threshold))
    mlflow.log_metric("f1_tuned", float(optimal_f1))

## 10. SHAP Values

SHAP is computed here (not deferred) because the values feed directly into the LLM prompt on Day 4.
Background sample and feature column order are logged as MLflow artifacts — the Day 4 LLM layer
needs both to reconstruct the explainer at inference time.

In [ ]:
# ── Background sample (always logged — Day 4 needs it for either model type) ──
background_sample = X_train.sample(
    n=min(100, len(X_train)), random_state=42)

# ── Explainer: TreeExplainer for DecisionTree, LinearExplainer for LogReg ──
if best_name == "DecisionTree":
    explainer = shap.TreeExplainer(best_model)
else:
    explainer = shap.LinearExplainer(best_model, masker=background_sample)

# Compute SHAP on a test sample
shap_sample_size = min(500, len(X_test))
X_shap = X_test.iloc[:shap_sample_size]
shap_values = explainer.shap_values(X_shap)

# For binary classifiers, shap_values may be [class_0_array, class_1_array]
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # class 1 = fraud/anomaly

# ── Global feature importance ──
shap_importance = pd.DataFrame({
    "feature": X_train.columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

print("Top 15 SHAP Features:")
print(shap_importance.head(15).to_string(index=False))

# ── Summary plot ──
shap.summary_plot(shap_values, X_shap, max_display=15, show=True)

## 11. Log Artifacts & Register Model

In [ ]:
with mlflow.start_run(run_id=best_run_id):

    # ── Background sample for SHAP reconstruction at inference ──
    background_sample.to_parquet("/tmp/shap_background.parquet", index=False)
    mlflow.log_artifact("/tmp/shap_background.parquet")

    # ── Feature column order (critical for Day 4 feature alignment) ──
    with open("/tmp/feature_columns.json", "w") as f:
        json.dump(list(X_train.columns), f)
    mlflow.log_artifact("/tmp/feature_columns.json")

    # ── SHAP importance table ──
    shap_importance.to_csv("/tmp/shap_feature_importance.csv", index=False)
    mlflow.log_artifact("/tmp/shap_feature_importance.csv")

    # ── OHE encoder (needed to encode categoricals at inference) ──
    import pickle
    with open("/tmp/ohe_encoder.pkl", "wb") as f:
        pickle.dump(ohe, f)
    mlflow.log_artifact("/tmp/ohe_encoder.pkl")

print("Artifacts logged: shap_background.parquet, feature_columns.json, "
      "shap_feature_importance.csv, ohe_encoder.pkl")

# ── Register best model ──
model_uri = f"runs:/{best_run_id}/model"
registered = mlflow.register_model(model_uri, "denial-risk-model")
print(f"\nRegistered model: {registered.name}  |  version: {registered.version}")

## 12. Save Provider Lookup Table

This table resolves provider-level features at inference time (Day 4, Test Case 4).
It is pre-computed from training data — no live recomputation over all claims.

In [ ]:
# Save as Delta for Databricks access
provider_stats.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("capstone.bronze.provider_lookup")

# Also save as parquet artifact in MLflow for portability
provider_stats.toPandas().to_parquet("/tmp/provider_lookup_table.parquet", index=False)
with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact("/tmp/provider_lookup_table.parquet")

print(f"Provider lookup saved: {provider_stats.count():,} providers")
provider_stats.show(5)

## 13. Summary

In [ ]:
print("=" * 65)
print("DAY 3 — ML MODELING COMPLETE")
print("=" * 65)
print(f"Best model:         {best_name}")
print(f"ROC-AUC:            {max(lr_roc_auc, dt_roc_auc):.4f}")
print(f"PR-AUC:             {max(lr_pr_auc, dt_pr_auc):.4f}")
print(f"F1 (default 0.5):   {results['F1 (0.5 cutoff)'].max():.4f}")
print(f"F1 (tuned @ {optimal_threshold:.2f}):  {optimal_f1:.4f}")
print(f"Optimal threshold:  {optimal_threshold:.2f}")
print(f"MLflow run ID:      {best_run_id}")
print(f"Registered model:   denial-risk-model v{registered.version}")
print(f"Features:           {X_train.shape[1]}")
print(f"Train claims:       {len(X_train):,}")
print(f"Test claims:        {len(X_test):,}")
print("-" * 65)
print("Artifacts logged to MLflow:")
print("  • shap_background.parquet     (SHAP explainer reconstruction)")
print("  • feature_columns.json        (feature order for inference)")
print("  • shap_feature_importance.csv (global feature ranking)")
print("  • ohe_encoder.pkl             (categorical encoder)")
print("  • provider_lookup_table.parquet")
print("-" * 65)
print("\nReady for Day 4 — LLM Orchestrator Layer")